In [15]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
import csv

# Imports centralisés — utilisez cette cellule pour gérer les dépendances


In [16]:
# Charger le second fichier avec les colonnes du CSV

df_epargne = pd.read_csv('Epargne_salariale.csv', sep=';')
df_epargne.head()

,Date,Montant (€)
0,Déjà disponible,"3 885,50"
1,01/06/2027,"1 804,21"
2,01/06/2028,"2 904,85"
3,01/06/2029,"6 108,18"
4,01/06/2030,"4 020,22"


In [ ]:
# Préparer les données ETF

# valeurs initiales
initial_amount = 30000
start_date = pd.Timestamp('2024-02-05')
end_date = pd.Timestamp('2037-12-05')

# Générer une série mensuelle sur le jour 5 de chaque mois
monthly_starts = pd.date_range(start=start_date, end=end_date, freq='MS')
monthly_dates = monthly_starts + pd.DateOffset(days=start_date.day - 1)
# s'assurer que la date de début et de fin sont incluses
if monthly_dates[0] != start_date:
    monthly_dates = monthly_dates.insert(0, start_date)
if monthly_dates[-1] != end_date:
    monthly_dates = monthly_dates.append(pd.DatetimeIndex([end_date]))
monthly_dates = pd.DatetimeIndex(sorted(set(monthly_dates)))

rates = [0.06, 0.08, 0.10, 0.12, 0.14, 0.16]

rows = []
for rate in rates:
    values = [initial_amount]
    for i in range(1, len(monthly_dates)):
        prev_date = monthly_dates[i - 1]
        curr_date = monthly_dates[i]
        months = (curr_date.year - prev_date.year) * 12 + (curr_date.month - prev_date.month)
        # avec nos dates sur le même jour du mois, months vaudra 1 pour pas mal d'étapes
        values.append(values[-1] * (1 + rate / 12) ** months)
    rows.append(pd.DataFrame({
        'Date': monthly_dates,
        'Montant': values,
        'Taux': [f'{rate * 100:.0f}%'] * len(monthly_dates)
    }))

etf_growth = pd.concat(rows, ignore_index=True)

months_total = (end_date.year - start_date.year) * 12 + (end_date.month - start_date.month)
summary = pd.DataFrame({
    'Taux': [f'{rate * 100:.0f}%' for rate in rates],
    'Montant final (€)': [round(initial_amount * (1 + rate / 12) ** months_total, 2) for rate in rates],
    'Gain (€)': [round(initial_amount * (1 + rate / 12) ** months_total - initial_amount, 2) for rate in rates]
})

summary

,Taux,Montant final (€),Gain (€)
0,6%,68657.42,38657.42
1,8%,90395.23,60395.23
2,10%,118961.35,88961.35
3,12%,156483.77,126483.77
4,14%,205748.37,175748.37
5,16%,270400.72,240400.72


In [ ]:
# Préparer le capital restant dû
amort_curve = df_amort[['Date', 'capital restant dû']].copy()
amort_curve['Date'] = pd.to_datetime(amort_curve['Date'], dayfirst=True)
amort_curve = amort_curve.sort_values('Date')
amort_curve['capital restant dû'] = pd.to_numeric(
    amort_curve['capital restant dû'].astype(str).str.replace(' ', '').str.replace(',', '.'),
    errors='coerce'
)

# Préparer etf_growth
etf_growth['Date'] = pd.to_datetime(etf_growth['Date'])

# Graphique comparatif
fig = px.line(
    etf_growth,
    x='Date',
    y='Montant',
    color='Taux',
    labels={'Date': 'Date', 'Montant': 'Montant (€)', 'Taux': 'Taux de rendement'},
    title='Évolution mensuelle des ETF et du capital restant dû',
    height=700
)

# Fond noir et texte en blanc pour lisibilité
fig.update_layout(
    paper_bgcolor='black',
    plot_bgcolor='black',
    font_color='white',
)

# Axes et grilles lisibles sur fond sombre
fig.update_xaxes(showgrid=True, gridcolor='gray', tickfont_color='white', title_font_color='white')
fig.update_yaxes(showgrid=True, gridcolor='gray', tickfont_color='white', title_font_color='white')

# Ajouter la courbe du capital restant dû
fig.add_scatter(
    x=amort_curve['Date'],
    y=amort_curve['capital restant dû'],
    mode='lines',
    name='Capital restant dû',
    line=dict(color='white', width=3)
)

# Construire un index mensuel correspondant aux dates d'etf_growth
monthly_index = pd.DatetimeIndex(sorted(etf_growth['Date'].unique()))
# Interpoler le capital restant dû sur ces dates
amort_series = amort_curve.set_index('Date')['capital restant dû']
amort_on_months = amort_series.reindex(monthly_index).interpolate(method='time')

# Récupérer les couleurs assignées par plotly pour chaque trace ETF
color_map = {}
for trace in fig.data:
    name = trace.name
    # seules les traces ETF ont des noms correspondant aux taux (ex: '6%')
    if isinstance(name, str) and name.endswith('%'):
        c = None
        if hasattr(trace, 'line') and getattr(trace.line, 'color', None) is not None:
            c = trace.line.color
        color_map[name] = c

# Calculer intersections (première date où ETF >= capital restant dû) et annoter
annotations = []
for taux, grp in etf_growth.groupby('Taux'):
    grp_sorted = grp.sort_values('Date').set_index('Date')
    etf_vals = grp_sorted['Montant']
    amort_vals = amort_on_months.reindex(grp_sorted.index).interpolate(method='time')
    mask = (etf_vals >= amort_vals).fillna(False)
    if mask.any():
        inter_date = mask.idxmax()
        inter_val = float(etf_vals.loc[inter_date])
        color = color_map.get(taux, 'yellow')
        # ajouter un marqueur avec la même couleur que la courbe si disponible
        fig.add_scatter(x=[inter_date], y=[inter_val], mode='markers', marker=dict(size=10, color=color), name=f'Intersection {taux}')
        # annotation texte avec date
        annotations.append(dict(
            x=inter_date,
            y=inter_val,
            xanchor='left',
            yanchor='bottom',
            text=f"{taux} — {inter_date.strftime('%d/%m/%Y')}",
            font=dict(color='white'),
            showarrow=True,
            arrowcolor=color
        ))

# Ajouter une ligne verticale pointillée rouge au 05/09/2031
vline_date = pd.Timestamp('2031-09-05')
fig.add_vline(x=vline_date, line=dict(color='red', width=2, dash='dot'))
# Annotation pour la ligne verticale
annotations.append(dict(
    x=vline_date,
    y=1.02 * etf_growth['Montant'].max(),
    xanchor='left',
    yanchor='bottom',
    text='Entrée Rose et Célestin supérieur — 05/09/2031',
    font=dict(color='red'),
    showarrow=False
))

fig.update_layout(annotations=annotations)

fig.show()